In [ ]:
# parameters
# BINDER_FAST: set N=4, n_eps=6 for fast cloud execution
N = 8                 # Hilbert space truncation
omega0_GHz = 5.0      # bare 0->1 frequency (GHz)
K_GHz = 0.2           # Kerr nonlinearity (GHz)
kappa_MHz = 1.0       # peak single-photon loss rate (MHz)
Gamma_MHz = 100.0     # bath FWHM (MHz)
nbar = 0.02           # thermal occupation
gamma_phi_MHz = 0.05  # pure dephasing rate (MHz)
epsilon_GHz = 0.05    # representative drive amplitude (GHz)
n_eps = 15            # number of points in epsilon sweep

In [ ]:
# hide
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
%matplotlib widget

from bosonic_gates.driven_kerr import DrivenKerrConfig, run_lindblad, run_redfield
from bosonic_gates.error_budget import compute_error_budget, ErrorBudget
from bosonic_gates.error_budget.budget import compute_error_budget_sweep

TWO_PI = 2 * np.pi

## Module 6b: Systematic Error Budget Computation

**Learning objectives:**
- Apply the turn-one-on-at-a-time methodology to compute per-channel infidelity contributions
- Evaluate the error budget at a single operating point and interpret the bar chart
- Sweep drive amplitude and observe how each channel's contribution evolves
- Compare Lindblad and Redfield budgets and understand their small differences
- Understand when the additive budget approximation is valid (total infidelity ≲ 10 %)

---

**Sections:**
[1 Turn-One-On-at-a-Time Methodology](#sec1) ·
[2 Single-Point Budget](#sec2) ·
[3 Budget Sweep vs Drive Amplitude](#sec3) ·
[4 Comparing Methods](#sec4) ·
[5 Exercises](#sec5)

<a id="sec1"></a>
## 1  Turn-One-On-at-a-Time Methodology

The **turn-one-on-at-a-time** approach is the standard method for decomposing a
total error into per-channel contributions without cross-term ambiguity.

**Protocol:**
1. Run the full simulation with all channels active → baseline infidelity $1 - F_{\rm all}$.
2. Disable channel $i$ (set its rate to zero) and re-run → $1 - F_{-i}$.
3. The per-channel contribution is:

$$\Delta\mathcal{I}_i = (1-F_{\rm all}) - (1-F_{-i}) = F_{-i} - F_{\rm all}.$$

**Mathematical justification:**
The Lindblad equation is linear in the jump operators. To first order in the
rates, the state $\rho(t) = \rho_0(t) + \sum_i \kappa_i \rho_i^{(1)}(t) + O(\kappa^2)$.
Therefore the infidelity decomposes additively:

$$1 - F_{\rm total} = \sum_i \Delta\mathcal{I}_i + O(\kappa_i \kappa_j).$$

**Validity condition:** The cross-terms $O(\kappa_i \kappa_j)$ are negligible when
the *total* infidelity is small ($\lesssim 10\%$). At higher infidelities the
per-channel contributions are still useful diagnostics, but the sum no longer
equals the total exactly.

**Channels tracked in `compute_error_budget`:**

| Channel | Disabled by setting | Jump operator |
|---|---|---|
| `photon_loss` | $\kappa \to 0$ | $\sqrt{\kappa(1+\bar{n})}\hat{a}$ |
| `thermal` | $\bar{n} \to 0$ | $\sqrt{\kappa\bar{n}}\hat{a}^\dagger$ |
| `dephasing` | $\gamma_\phi \to 0$ | $\sqrt{\gamma_\phi}\hat{n}$ |

*Lab note: this approach requires 4 simulations (one full + one per channel). For
large Hilbert spaces or long simulations this can be expensive. A trick: run the
budget on a coarse grid first, then refine only where the budget indicates a
dominant channel is changing rapidly.*

<a id="sec2"></a>
## 2  Single-Point Error Budget

In [ ]:
# Build canonical configuration
cfg = DrivenKerrConfig(
    N=N,
    omega0=TWO_PI * omega0_GHz,
    K=TWO_PI * K_GHz,
    omega_d=TWO_PI * (omega0_GHz - 0.005),   # 5 MHz red-detuned
    kappa=TWO_PI * kappa_MHz * 1e-3,
    Gamma=TWO_PI * Gamma_MHz * 1e-3,
    nbar=nbar,
    gamma_phi=TWO_PI * gamma_phi_MHz * 1e-3,
    k_max=5,
    n_t=512,
)

# Representative drive amplitude
cfg_driven = cfg.replace(epsilon=TWO_PI * epsilon_GHz)
print(f"Drive amplitude ε/2π = {cfg_driven.epsilon/TWO_PI:.4f} GHz = "
      f"{cfg_driven.epsilon/cfg.K:.2f} K")

# Time array: 5 decay times at κ
tlist = np.linspace(0, 5.0 / cfg.kappa, 200)
print(f"Simulation duration: {tlist[-1]*cfg.kappa:.1f} / κ = {tlist[-1]*1e6:.2f} μs")
print()
print("Running error budget (4 Lindblad simulations)...")

budget = compute_error_budget(cfg_driven, tlist, method="lindblad")
print(budget)

In [ ]:
# Bar chart of the single-point budget
d = budget.as_dict()
channel_names = ["photon_loss", "thermal", "dephasing"]
channel_labels = ["Photon loss", "Thermal", "Dephasing"]
channel_vals = [d[k] for k in channel_names]
total = d["total"]
floor = 1e-6  # display floor for log scale

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left panel: bar chart
ax = axes[0]
x = np.arange(len(channel_names))
colors = ["C0", "C1", "C2"]
bars = ax.bar(x, [max(v, floor) for v in channel_vals], color=colors, alpha=0.85)

# Annotate with percentages
for i, (val, bar_obj) in enumerate(zip(channel_vals, bars)):
    pct = val / total * 100 if total > 0 else 0
    ax.text(bar_obj.get_x() + bar_obj.get_width()/2,
            max(val, floor) * 1.1,
            f"{pct:.1f}%",
            ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.axhline(total, color="k", ls="--", lw=1.5, label=f"Total = {total:.4f}")
ax.set_xticks(x)
ax.set_xticklabels(channel_labels)
ax.set_ylabel("Infidelity contribution (1 - F)")
ax.set_title(f"Error budget (Method A, Lindblad)\n"
             f"ε/2π = {cfg_driven.epsilon/TWO_PI:.4f} GHz = {cfg_driven.epsilon/cfg.K:.2f}K")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

# Right panel: log-scale version for clearer comparison
ax2 = axes[1]
ax2.bar(x, [max(v, floor) for v in channel_vals], color=colors, alpha=0.85)
for i, val in enumerate(channel_vals):
    pct = val / total * 100 if total > 0 else 0
    ax2.text(i, max(val, floor) * 1.5, f"{val:.2e}\n({pct:.1f}%)",
             ha="center", va="bottom", fontsize=8)

ax2.set_yscale("log")
ax2.set_ylim(floor * 0.1, total * 10)
ax2.axhline(total, color="k", ls="--", lw=1.5, label=f"Total = {total:.2e}")
ax2.set_xticks(x)
ax2.set_xticklabels(channel_labels)
ax2.set_ylabel("Infidelity (log scale)")
ax2.set_title("Same — log scale for detail")
ax2.legend()
ax2.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSummary table:")
print(f"  {'Channel':<15} {'Infidelity':>12}  {'Fraction':>10}")
print(f"  {'-'*40}")
for name, val in zip(channel_labels, channel_vals):
    pct = val / total * 100 if total > 0 else 0
    print(f"  {name:<15} {val:>12.4e}  {pct:>9.1f}%")
print(f"  {'Total':<15} {total:>12.4e}  {'100.0%':>10}")

<a id="sec3"></a>
## 3  Budget Sweep vs Drive Amplitude

We now sweep the drive amplitude $\varepsilon$ from very small (weak drive) to
$\varepsilon \sim K$ (strong drive) and compute the full error budget at each point.

**Expected behaviour (Method A, Lindblad):**
- Loss dominates at all drive amplitudes because $\kappa \gg \kappa\bar{n}$ and
  $\kappa \gg \gamma_\phi$ for our parameters.
- Method A is *insensitive to drive amplitude* — all rates are fixed at bare $\omega_0$.
  The total infidelity therefore grows monotonically with the simulation time
  ($t \propto 1/\varepsilon$ for a fixed number of oscillations), not with $\varepsilon$.

**What Method C (Floquet–Markov) shows differently:**
- The loss rate drops as $\varepsilon$ increases (sideband shift suppresses bath coupling).
- A drive-induced dephasing channel appears that is absent in Method A.
- See Module 3c for the full Floquet-Markov computation.

In [ ]:
# Sweep epsilon and compute error budget at each point
eps_arr = np.geomspace(TWO_PI * 1e-4, TWO_PI * 0.2, n_eps)  # 0.1 MHz to 200 MHz
tlist_sweep = np.linspace(0, 5.0 / cfg.kappa, 100)

print(f"Running budget sweep over {n_eps} drive amplitudes ...")
print(f"  ε/2π range: {eps_arr[0]/TWO_PI*1e3:.3f} to {eps_arr[-1]/TWO_PI*1e3:.0f} MHz")
print(f"  Simulation time: {tlist_sweep[-1]*cfg.kappa:.1f}/κ per point")

sweep = compute_error_budget_sweep(cfg, eps_arr, tlist_sweep, method="lindblad")

print("Sweep complete.")
print(f"\nTotal infidelity range: {sweep['total'].min():.4e} to {sweep['total'].max():.4e}")

In [ ]:
# Stacked area chart of the budget sweep
eps_over_K = eps_arr / cfg.K

total    = sweep["total"]
loss     = np.maximum(sweep["photon_loss"], 0)
thermal  = np.maximum(sweep["thermal"], 0)
dephasing = np.maximum(sweep["dephasing"], 0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Panel (a): stacked area
ax = axes[0]
ax.stackplot(
    eps_over_K,
    loss, thermal, dephasing,
    labels=["Photon loss", "Thermal", "Dephasing"],
    colors=["C0", "C1", "C2"], alpha=0.75,
)
ax.plot(eps_over_K, total, "k-", lw=2, label="Total")
ax.set_xlabel(r"Drive amplitude $\varepsilon / K$")
ax.set_ylabel("Infidelity contribution (1 - F)")
ax.set_title("Method A (Lindblad): Error budget vs drive amplitude")
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)

# Panel (b): log-log to see small channels
ax = axes[1]
ax.loglog(eps_over_K, total, "k-", lw=2, label="Total")
ax.loglog(eps_over_K, np.where(loss > 1e-10, loss, np.nan),
          "C0-o", ms=4, lw=1.5, label="Photon loss")
ax.loglog(eps_over_K, np.where(thermal > 1e-10, thermal, np.nan),
          "C1-s", ms=4, lw=1.5, label="Thermal")
ax.loglog(eps_over_K, np.where(dephasing > 1e-10, dephasing, np.nan),
          "C2-^", ms=4, lw=1.5, label="Dephasing")
ax.set_xlabel(r"Drive amplitude $\varepsilon / K$")
ax.set_ylabel("Infidelity (log scale)")
ax.set_title("Same — log-log for small channels")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, which="both")

plt.tight_layout()
plt.show()

print("Key observation (Method A):")
print("  Loss channel dominates at all drive amplitudes.")
print("  Method A rates are independent of ε — budget is flat (not sweep-dependent).")
print()
print("Lab note: in Method C (Floquet-Markov), dephasing becomes drive-dependent")
print("  and loss is SUPPRESSED at strong drive. See Module 3c for details.")

<a id="sec4"></a>
## 4  Comparing Methods A and B (Lindblad vs Redfield)

Method B (Bloch–Redfield) samples the bath at sideband frequencies
$\omega_0 + k\omega_d$ (with $k = 0, \pm 1, \pm 2, \ldots$), but using
the *undriven* Fock-state coupling matrix elements (not the dressed Floquet modes).

For our parameters ($\Gamma/2\pi = 100$ MHz bath width, $\omega_d/2\pi \approx 5$ GHz),
the sidebands at $k \neq 0$ are separated by $\omega_d \gg \Gamma$ from the bath
center. The bath strongly suppresses coupling to $k \neq 0$ sidebands.
As a result, Method B gives nearly the same result as Method A for our parameters.

**Method C (Floquet–Markov)** uses the *dressed* quasi-energy differences and
dressed coupling matrix elements — capturing the drive-induced sideband shift.
This is the method that shows the 20× discrepancy at $\varepsilon = K$
(see Module 3c). For quantitatively correct error budgets in driven systems,
always use Method C.

*Lab note: a good rule of thumb — if your operating drive amplitude is
$\varepsilon/2\pi \gtrsim 0.1\,\Gamma/2\pi$, use Floquet–Markov.*

In [ ]:
# Compare Lindblad and Redfield budgets at the single representative point
print("Running Redfield budget at the same operating point ...")

budget_A = budget  # already computed above (Lindblad)
budget_B = compute_error_budget(cfg_driven, tlist, method="redfield")

print("\nMethod A (Lindblad):")
print(budget_A)
print()
print("Method B (Redfield):")
print(budget_B)

In [ ]:
# Side-by-side bar chart
channel_keys = ["photon_loss", "thermal", "dephasing"]
channel_labels_short = ["Loss", "Thermal", "Dephasing"]

vals_A = [budget_A.as_dict()[k] for k in channel_keys]
vals_B = [budget_B.as_dict()[k] for k in channel_keys]
total_A = budget_A.total_infidelity
total_B = budget_B.total_infidelity
floor = 1e-7

x = np.arange(len(channel_keys))
w = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, [max(v, floor) for v in vals_A], w,
       color="C0", alpha=0.85, label=f"A (Lindblad), total={total_A:.3e}")
ax.bar(x + w/2, [max(v, floor) for v in vals_B], w,
       color="C1", alpha=0.85, label=f"B (Redfield), total={total_B:.3e}")

ax.set_yscale("log")
ax.set_ylim(floor * 0.1, max(total_A, total_B) * 10)
ax.set_xticks(x)
ax.set_xticklabels(channel_labels_short)
ax.set_ylabel("Infidelity contribution (log scale)")
ax.set_title(f"Method A vs Method B: error budget comparison\n"
             f"ε/2π = {cfg_driven.epsilon/TWO_PI:.4f} GHz = {cfg_driven.epsilon/cfg.K:.2f}K")
ax.legend(fontsize=9)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nMethod A total infidelity: {total_A:.4e}")
print(f"Method B total infidelity: {total_B:.4e}")
print(f"Relative difference: {abs(total_A - total_B)/total_A:.1%}")
print()
print("Methods A and B agree closely for our narrow bath (Γ ≪ ω_d).")
print("Method C (Floquet-Markov) would show a much larger discrepancy")
print("at ε ≳ Γ/2π. Cross-reference nb3c_floquet_markov for the full picture.")

<a id="sec5"></a>
## 5  Exercises

### Exercise 1: Find the 1% infidelity threshold

For Method A, at what drive amplitude $\varepsilon/2\pi$ does the total infidelity
first reach 1%? Find it by interpolating the sweep results, then verify with a
direct simulation at that amplitude.

In [ ]:
# YOUR CODE HERE
# Hint: use np.interp to find the epsilon where sweep['total'] == 0.01
# Note: sweep['total'] may be monotone or not — check the shape first
# sweep['epsilon'] is the array of ε values in rad/s

### Exercise 2: Remove thermal channel

Repeat the budget sweep with `nbar=0.0`. Which channel disappears from the bar
chart? By how much does the total infidelity change? Quantify the thermal
correction as a fraction of the total budget at $\varepsilon = 0.5K$.

In [ ]:
# YOUR CODE HERE
# Hint:
# cfg_zero_nbar = cfg.replace(nbar=0.0)
# sweep_zero_nbar = compute_error_budget_sweep(cfg_zero_nbar, eps_arr, tlist_sweep)
# Then compare sweep_zero_nbar['thermal'] to sweep['thermal']